In [0]:
from pyspark.sql.functions import sum as _sum, count, countDistinct

silver_path = "/Volumes/workspace/retail_raw/silver/"
gold_path = "/Volumes/workspace/retail_raw/gold/"

orders = spark.read.format("delta").load(silver_path + "orders")
order_items = spark.read.format("delta").load(silver_path + "order_items")
customers = spark.read.format("delta").load(silver_path + "customers")
products = spark.read.format("delta").load(silver_path + "products")

# 1. Daily Revenue
daily_revenue = (
    orders.join(order_items, "order_id")
    .groupBy("order_purchase_date")
    .agg(
        _sum("price").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy("order_purchase_date")
)

daily_revenue.write.format("delta").mode("overwrite").save(gold_path + "daily_revenue")
print(f"Gold daily_revenue: {daily_revenue.count()} rows")
daily_revenue.show(5)

Gold daily_revenue: 616 rows
+-------------------+-----------------+------------+
|order_purchase_date|    total_revenue|total_orders|
+-------------------+-----------------+------------+
|         2016-09-04|            72.89|           1|
|         2016-09-05|             59.5|           1|
|         2016-09-15|           134.97|           1|
|         2016-10-02|            100.0|           1|
|         2016-10-03|463.4800000000001|           8|
+-------------------+-----------------+------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, sum as _sum, count, countDistinct
revenue_by_state = (
    orders.join(order_items, "order_id")
    .join(customers, "customer_id")
    .groupBy("customer_state")
    .agg(
        _sum("price").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy(col("total_revenue").desc())
)

revenue_by_state.write.format("delta").mode("overwrite").save(gold_path + "revenue_by_state")

print(f"Gold revenue_by_state: {revenue_by_state.count()} rows")
revenue_by_state.show(10)

Gold revenue_by_state: 27 rows
+--------------+------------------+------------+
|customer_state|     total_revenue|total_orders|
+--------------+------------------+------------+
|            SP| 5202955.049999566|       41375|
|            RJ|1824092.6700000283|       12762|
|            MG|1585308.0300000214|       11544|
|            RS| 750304.0199999918|        5432|
|            PR| 683083.7599999926|        4998|
|            SC|  520553.339999996|        3612|
|            BA|511349.98999999685|        3358|
|            DF| 302603.9399999997|        2125|
|            GO| 294591.9499999999|        2007|
|            ES| 275037.3099999999|        2025|
+--------------+------------------+------------+
only showing top 10 rows


In [0]:
top_categories = (
    order_items.join(products, "product_id")
    .groupBy("product_category_name")
    .agg(
        _sum("price").alias("total_revenue"),
        count("order_id").alias("total_items_sold")
    )
    .orderBy(col("total_revenue").desc())
)

top_categories.write.format("delta").mode("overwrite").save(gold_path + "top_categories")

print(f"Gold top_categories: {top_categories.count()} rows")
top_categories.show(10)

Gold top_categories: 74 rows
+---------------------+------------------+----------------+
|product_category_name|     total_revenue|total_items_sold|
+---------------------+------------------+----------------+
|         beleza_saude|1258681.3399999673|            9670|
|   relogios_presentes|1205005.6799999988|            5991|
|      cama_mesa_banho|1036988.6800000715|           11115|
|        esporte_lazer| 988048.9700000416|            8641|
| informatica_acess...| 911954.3200000357|            7827|
|     moveis_decoracao|  729762.490000042|            8334|
|           cool_stuff| 635290.8500000002|            3796|
| utilidades_domest...| 632248.6600000212|            6964|
|           automotivo| 592720.1100000112|            4235|
|   ferramentas_jardim| 485256.4600000133|            4347|
+---------------------+------------------+----------------+
only showing top 10 rows
